# Statistical Significance Analysis

Addresses Reviewer Comment 3: provides mean, standard deviation, and p-values comparing the soft-voting ensemble against each individual base model using a **paired t-test** over 10-fold CV F1-scores.

In [ ]:
%pip install -q openpyxl catboost xgboost imbalanced-learn pymrmr kagglehub scipy

In [ ]:
import os
import subprocess

os.environ["CC"] = "/opt/homebrew/Cellar/gcc/14.2.0_1/bin/gcc-14"
result = subprocess.run(["ls", "-la", "/opt/homebrew/Cellar/gcc/14.2.0_1/bin/"],
                        capture_output=True, text=True)
if "g++-14" in result.stdout:
    os.environ["CXX"] = "/opt/homebrew/Cellar/gcc/14.2.0_1/bin/g++-14"
else:
    print("g++-14 not found in the expected directory!")

In [ ]:
import json
import numpy as np
import pandas as pd
import kagglehub
from scipy import stats

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import BaggingClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import pymrmr

## 1. Load & Preprocess Data

In [ ]:
path = kagglehub.dataset_download("luzrello/dyslexia")
df_desktop = pd.read_csv(path + "/Dyt-desktop.csv", delimiter=';')

# Label encoding categorical columns in df_desktop
categorical_cols = df_desktop.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df_desktop[col] = le.fit_transform(df_desktop[col])

# Finding duplicate samples
duplicates = df_desktop.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
    print("Dropping Duplicates...")
    df_desktop.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
else:
    print("No Duplicate Samples Found.")

X = df_desktop.drop("Dyslexia", axis=1).reset_index(drop=True)
y = df_desktop['Dyslexia'].reset_index(drop=True)

print(f"\nDataset shape: {X.shape}  |  Classes: {y.value_counts().to_dict()}")

In [ ]:
classes, samples_in_class = np.unique(y, return_counts=True)
n_classes     = len(classes)
total_samples = len(y)
class_weights      = [float(total_samples / (n_classes * count)) for count in samples_in_class]
class_weights_dict = {i: class_weights[i] for i in range(n_classes)}

negative_samples = samples_in_class[0]
positive_samples = samples_in_class[1]
scale_pos_weight  = float(negative_samples / positive_samples)

NUM_FEATURES = 165

## 2. mRMR Feature Selection (cached)

In [ ]:
MRMR_CACHE = "mrmr_selected_features.json"

if os.path.exists(MRMR_CACHE):
    with open(MRMR_CACHE) as f:
        selected_features = json.load(f)
    print(f"Loaded {len(selected_features)} features from cache.", flush=True)
else:
    print("Running mRMR on full dataset (this will take ~30 min)...", flush=True)
    data_full = pd.concat([y.rename('target'), X], axis=1)
    data_full.columns = data_full.columns.astype(str)
    selected_features = pymrmr.mRMR(data_full, 'MIQ', NUM_FEATURES)
    with open(MRMR_CACHE, 'w') as f:
        json.dump(selected_features, f)
    print(f"mRMR done. Features cached to {MRMR_CACHE}", flush=True)

X_sel = X[selected_features]

## 3. 10-Fold CV — Record Per-Fold F1 for Each Model and the Ensemble

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

model_names = ['XGBoost', 'Bagging', 'HistGradBoost', 'CatBoost', 'Ensemble']
fold_f1     = {name: [] for name in model_names}
fold_acc    = {name: [] for name in model_names}
fold_prec   = {name: [] for name in model_names}
fold_rec    = {name: [] for name in model_names}

for fold, (train_index, test_index) in enumerate(skf.split(X_sel, y), 1):
    print(f"Fold {fold:2d} | Training...", flush=True)

    X_train, X_test = X_sel.iloc[train_index], X_sel.iloc[test_index]
    y_train, y_test = y.iloc[train_index],     y.iloc[test_index]

    # Individual models — train and evaluate separately
    individual_models = [
        ('XGBoost',       XGBClassifier(verbosity=0, scale_pos_weight=scale_pos_weight, random_state=42)),
        ('Bagging',       BaggingClassifier(estimator=DecisionTreeClassifier(class_weight='balanced'), random_state=42)),
        ('HistGradBoost', HistGradientBoostingClassifier(class_weight=class_weights_dict, random_state=42)),
        ('CatBoost',      CatBoostClassifier(verbose=0, class_weights=class_weights, random_state=42)),
    ]

    fitted_models = []
    for name, model in individual_models:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        fold_f1[name].append(f1_score(y_test, preds, average='weighted'))
        fold_acc[name].append(accuracy_score(y_test, preds))
        fold_prec[name].append(precision_score(y_test, preds, average='weighted'))
        fold_rec[name].append(recall_score(y_test, preds, average='weighted'))
        fitted_models.append((name.lower().replace(' ', '_'), model))

    # Ensemble (soft voting)
    ensemble_models = [
        ('xgb', XGBClassifier(verbosity=0, scale_pos_weight=scale_pos_weight, random_state=42)),
        ('bg',  BaggingClassifier(estimator=DecisionTreeClassifier(class_weight='balanced'), random_state=42)),
        ('gbc', HistGradientBoostingClassifier(class_weight=class_weights_dict, random_state=42)),
        ('cat', CatBoostClassifier(verbose=0, class_weights=class_weights, random_state=42)),
    ]
    ensemble = VotingClassifier(estimators=ensemble_models, voting='soft', weights=[1.0]*4)
    ensemble.fit(X_train, y_train)
    ens_preds = ensemble.predict(X_test)
    fold_f1['Ensemble'].append(f1_score(y_test, ens_preds, average='weighted'))
    fold_acc['Ensemble'].append(accuracy_score(y_test, ens_preds))
    fold_prec['Ensemble'].append(precision_score(y_test, ens_preds, average='weighted'))
    fold_rec['Ensemble'].append(recall_score(y_test, ens_preds, average='weighted'))

    ens_f1 = fold_f1['Ensemble'][-1]
    print(f"Fold {fold:2d} | Ensemble F1: {ens_f1:.4f}", flush=True)

print("\nDone.", flush=True)

## 4. Summary Statistics (Mean, Std, N)

In [ ]:
summary_rows = []
for name in model_names:
    f1_arr  = np.array(fold_f1[name])
    acc_arr = np.array(fold_acc[name])
    pr_arr  = np.array(fold_prec[name])
    rc_arr  = np.array(fold_rec[name])
    summary_rows.append({
        'Model':        name,
        'N_Folds':      len(f1_arr),
        'F1_Mean':      round(f1_arr.mean(),  4),
        'F1_Std':       round(f1_arr.std(),   4),
        'Acc_Mean':     round(acc_arr.mean(), 4),
        'Acc_Std':      round(acc_arr.std(),  4),
        'Prec_Mean':    round(pr_arr.mean(),  4),
        'Prec_Std':     round(pr_arr.std(),   4),
        'Recall_Mean':  round(rc_arr.mean(),  4),
        'Recall_Std':   round(rc_arr.std(),   4),
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

## 5. Paired t-Test: Ensemble vs Each Individual Model

H₀: no difference in F1 between the ensemble and the baseline model.  
p < 0.05 → statistically significant improvement.

In [ ]:
ens_f1_arr = np.array(fold_f1['Ensemble'])

ttest_rows = []
for name in ['XGBoost', 'Bagging', 'HistGradBoost', 'CatBoost']:
    base_arr = np.array(fold_f1[name])
    t_stat, p_val = stats.ttest_rel(ens_f1_arr, base_arr)
    ttest_rows.append({
        'Comparison':       f'Ensemble vs {name}',
        'Ensemble_F1_Mean': round(ens_f1_arr.mean(), 4),
        'Baseline_F1_Mean': round(base_arr.mean(),   4),
        'Mean_Difference':  round((ens_f1_arr - base_arr).mean(), 4),
        't_statistic':      round(t_stat, 4),
        'p_value':          round(p_val,  6),
        'Significant':      'Yes' if p_val < 0.05 else 'No',
    })
    print(f"Ensemble vs {name:<15} | t={t_stat:.4f} | p={p_val:.6f} | {'Significant' if p_val < 0.05 else 'Not significant'}")

df_ttest = pd.DataFrame(ttest_rows)

## 6. Save Results to Excel

In [ ]:
# Per-fold F1 scores for all models
df_fold_f1 = pd.DataFrame(fold_f1)
df_fold_f1.index = [f'Fold_{i+1}' for i in range(10)]
df_fold_f1.index.name = 'Fold'

excel_path = "../../Plots & Visuals/Statistical_Analysis.xlsx"
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer,  index=False, sheet_name='Summary_Statistics')
    df_ttest.to_excel(writer,    index=False, sheet_name='Paired_T_Test')
    df_fold_f1.to_excel(writer,              sheet_name='Per_Fold_F1')

print(f"Saved to: {excel_path}")